In [1]:
from skyfield.api import EarthSatellite, load
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, IntSlider, Checkbox
import ipywidgets as widgets

# -----------------------------
# Compute circular footprint
# -----------------------------
def footprint_boundary(lat0, lon0, radius_km, n=360):
    lat0 = np.radians(lat0)
    lon0 = np.radians(lon0)
    R = 6371.0  # Earth radius (km)
    lats = []
    lons = []
    for bearing in np.linspace(0, 2*np.pi, n):
        lat = np.arcsin(
            np.sin(lat0) * np.cos(radius_km / R) +
            np.cos(lat0) * np.sin(radius_km / R) * np.cos(bearing))
        lon = lon0 + np.arctan2(
            np.sin(bearing) * np.sin(radius_km / R) * np.cos(lat0),
            np.cos(radius_km / R) - np.sin(lat0) * np.sin(lat))
        lats.append(np.degrees(lat))
        lons.append(np.degrees(lon))
    return lats, lons

def load_satellites(filename):
    satellites = []
    with open(filename, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    for i in range(0, len(lines), 3):
        name = lines[i]
        line1 = lines[i + 1]
        line2 = lines[i + 2]
        sat = EarthSatellite(line1, line2, name)
        satellites.append(sat)
    return satellites

ts = load.timescale()
t = ts.utc(2026, 5, 4, 8, 0, 0)
satellites = load_satellites("gps-ops.txt")
fig = go.Figure()
R = 6371.0
colors = [
    "cyan",
    "lime",
    "orange",
    "magenta",
    "yellow",
    "red",
    "white"]
for idx, sat in enumerate(satellites):
    geocentric = sat.at(t)
    subpoint = geocentric.subpoint()
    lat = subpoint.latitude.degrees
    lon = subpoint.longitude.degrees
    # Altitude above Earth surface
    alt = geocentric.distance().km - R
    # Footprint geometry
    ratio = R / (R + alt)
    ratio = np.clip(ratio, -1, 1)
    footprint_angle = np.arccos(ratio)
    footprint_radius_km = R * footprint_angle
    foot_lat, foot_lon = footprint_boundary(lat,lon,footprint_radius_km)
    color = colors[idx % len(colors)]
    # -------------------------
    # Footprint boundary
    # -------------------------
    fig.add_trace(go.Scattergeo(lat=foot_lat,lon=foot_lon, mode='lines',line=dict(color=color, width=2),name=f'{sat.name} Footprint'))
    # -------------------------
    # Satellite marker
    # -------------------------
    fig.add_trace(go.Scattergeo(lat=[lat],lon=[lon],mode='markers',marker=dict(size=7, color=color),name=sat.name,text=[sat.name]))
# -----------------------------
# Globe styling
# -----------------------------
fig.update_layout(
    title="GPS Satellite Footprints",
    geo=dict(
        projection_type="orthographic",
        showland=True,
        landcolor="rgb(40,40,40)",
        showocean=True,
        oceancolor="rgb(10,20,40)",
        showcoastlines=True,
        coastlinecolor="white",
        showcountries=True,
        countrycolor="gray",
        bgcolor="black"),
    paper_bgcolor="black",
    font=dict(color="white"),
    legend=dict(bgcolor="rgba(0,0,0,0.5)")
)
fig.show()

In [ ]:
R = 6371.0
colors = ["cyan","lime","orange","magenta","yellow","red","white"]

def footprint_boundary(lat0, lon0, radius_km, n=90):
    lat0, lon0 = np.radians(lat0), np.radians(lon0)
    lats, lons = [], []
    for b in np.linspace(0, 2*np.pi, n):
        lat = np.arcsin(
            np.sin(lat0)*np.cos(radius_km/R) +
            np.cos(lat0)*np.sin(radius_km/R)*np.cos(b))
        lon = lon0 + np.arctan2(
            np.sin(b)*np.sin(radius_km/R)*np.cos(lat0),
            np.cos(radius_km/R) - np.sin(lat0)*np.sin(lat))
        lats.append(np.degrees(lat))
        lons.append(np.degrees(lon))
    return lats, lons

def plot_satellites(year, month, day, hour, minute,
                    num_sats, show_footprints):
    t = ts.utc(year, month, day, hour, minute)
    fig = go.Figure()
    center_lat, center_lon = 0, 0
    for idx, sat in enumerate(satellites[:num_sats]):
        geo = sat.at(t)
        sub = geo.subpoint()
        lat, lon = sub.latitude.degrees, sub.longitude.degrees
        if idx == 0:
            center_lat, center_lon = lat, lon
        alt = geo.distance().km - R
        radius = R * np.arccos(np.clip(R/(R+alt), -1, 1))
        color = colors[idx % len(colors)]
        if show_footprints:
            foot_lat, foot_lon = footprint_boundary(lat, lon, radius)
            fig.add_trace(go.Scattergeo(
                lat=foot_lat, lon=foot_lon,
                mode='lines',
                line=dict(color=color, width=2),
                showlegend=False))

        fig.add_trace(go.Scattergeo(
            lat=[lat], lon=[lon],
            mode='markers',
            marker=dict(size=7, color=color),
            name=sat.name))

    fig.update_layout(
        height=750,
        paper_bgcolor="black",
        font=dict(color="white"),
        geo=dict(
            projection_type="orthographic",
            projection_rotation=dict(lon=center_lon, lat=center_lat),
            showland=True,
            landcolor="rgb(40,40,40)",
            showocean=True,
            oceancolor="rgb(10,20,40)",
            showcoastlines=True,
            coastlinecolor="white",
            bgcolor="black"))
    fig.show()

year = widgets.IntSlider(value=2026, min=2020, max=2030, description='Year', continuous_update=False)
month = widgets.IntSlider(value=5, min=1, max=12, description='Month', continuous_update=False)
day = widgets.IntSlider(value=11, min=1, max=31, description='Day', continuous_update=False)
hour = widgets.IntSlider(value=8, min=0, max=23, description='Hour', continuous_update=False)
minute = widgets.IntSlider(value=0, min=0, max=59, description='Minute', continuous_update=False)
num_sats = widgets.IntSlider(
    value=min(5, len(satellites)),
    min=1, max=len(satellites),
    description='Satellites',
    continuous_update=False)
show_footprints = widgets.Checkbox(value=True, description='Footprints')
ui = widgets.VBox([
    year, month, day,
    hour, minute,
    num_sats,
    show_footprints])
out = widgets.interactive_output(
    plot_satellites,
    {'year': year,
        'month': month,
        'day': day,
        'hour': hour,
        'minute': minute,
        'num_sats': num_sats,
        'show_footprints': show_footprints})

display(ui, out)

Output()